In [10]:
import pandas as pd
import numpy as np

In [11]:
df = pd.read_csv("../data/ote_prices_15min.csv", sep=";")

In [12]:
if df["price"].dtype == object:
    # If your file actually uses commas for decimals (e.g. 120,28), 
    df["price"] = df["price"].astype(str).str.replace(",", "")
df["price"] = pd.to_numeric(df["price"], errors="coerce")

In [13]:
raw_hour = df["time_interval"].str.split(":").str[0]
df["hour"] = raw_hour.str.replace(r"\D", "", regex=True).astype(int)

In [14]:
is_peak_hours = (df["hour"] >= 8) & (df["hour"] < 20)

In [15]:
df["Market Period"] = "Off-peak"
df.loc[is_peak_hours, "Market Period"] = "Peak"

In [16]:
df["day"] = pd.to_datetime(df["day"])

In [17]:
daily_results = []

for current_date, group in df.groupby("day"):
    peak_group = group[group["Market Period"] == "Peak"]
    offpeak_group = group[group["Market Period"] == "Off-peak"]
    
    base_price = group["price"].mean()
    peak_price = peak_group["price"].mean() if not peak_group.empty else np.nan
    offpeak_price = offpeak_group["price"].mean() if not offpeak_group.empty else np.nan
    
    daily_results.append({
        "Date": current_date.strftime("%Y-%m-%d"),
        "Baseload Price": round(base_price, 2) if pd.notna(base_price) else "",
        "Peakload Price": round(peak_price, 2) if pd.notna(peak_price) else "",
        "Off-peak Price": round(offpeak_price, 2) if pd.notna(offpeak_price) else ""
    })

In [18]:
result_df = pd.DataFrame(daily_results)
result_df.to_csv("../data/electricity_price_loads.csv", sep=";", index=False)